# Python primer — Week 13: Dictionaries

## Topics to cover
- Dictionaries: keys, values, and why they are useful
- Creating, modifying, and accessing dictionaries
- Dictionary methods and iteration
- Building dictionaries from biological data files
- Counting observations and nested dictionaries
- Bioinformatic examples using FASTA, FASTQ, population metadata, and GFF3

## Useful materials
- Haddock and Dunn, chapters 9 and parts of 11
- Slides from class
- [Python for Biologists: dictionaries](https://pythonforbiologists.com/tutorial/dictionaries.html)

## I. Introduction to dictionaries

Dictionaries (sometimes called associative arrays or hashes) store **key:value pairs**. We often want one piece of information directly connected to another: name → phone number, gene ID → sequence, population → latitude, or sample → read count.

Lists are accessed by position (`my_list[0]`), while dictionaries are accessed by **key** (`my_dict["Reno"]`). Keys are unique and must be hashable (strings, numbers, and tuples of immutable values are common). Values can be strings, numbers, lists, dictionaries, and other Python objects. Dictionaries preserve insertion order, but you should not rely on that order when you need alphabetical or numerical sorting. Key lookup is generally extremely fast.

In [1]:
cities = ["Reno", "Truckee", "Tucson"]
print(cities[0])               # list: use an index

Ctemp = {"Tucson": 95, "Truckee": 65, "Reno": 74}
print(Ctemp["Reno"])           # dictionary: use a key

Reno
74


## II. Creating and modifying dictionaries

We can create dictionaries by listing key:value pairs inside curly brackets. For longer dictionaries, spreading the pairs across multiple lines makes them easier to read. Python allows these line breaks inside `{}`.

Keep in mind that we are hardcoding dictionaries below to make toy examples. For most purposes, dictionaries are built on the fly while reading in larger amounts of real data. In other words, dont stress the details of notation in code boxes below.

In [ ]:
Pbook = { 'Ken Stephens':'2599855888','Mick Collins':'3544333321', 'Jen Miles':'9875842194' }

In [3]:
Pbook = { 
        'Ken Stephens':'2599855888', 
        'Mick Collins':'3544333321', 
        'Jen Miles':'9875842194' 
    }

In [9]:
print(Pbook)
print(Pbook["Mick Collins"])

Ctemp = {
    "Tucson": 95,
    "Truckee": 65,
    "Reno": 74,
    "Laramie": 50,
    "Flagstaff": 75,
    "Bozeman": 70,
}
print(Ctemp["Reno"])

{'Ken Stephens': '2599855888', 'Mick Collins': '3544333321', 'Jen Miles': '9875842194'}
3544333321
74


An empty dictionary is created using `{}`. Assigning a value to a new key adds an item; assigning a value to an existing key replaces the old value. A dictionary cannot have two separate entries with the same key.

In [10]:
populations = {}
populations["Reno"] = 25
populations["Truckee"] = 18
print(populations)

populations["Reno"] = 30   # update existing value
print(populations)
print(len(populations))      # number of key:value pairs

{'Reno': 25, 'Truckee': 18}
{'Reno': 30, 'Truckee': 18}
2


### Looking up missing keys

`d[key]` raises a `KeyError` if the key is absent. Use `in` to check first, or `.get()` when a missing key is expected.

In [11]:
print("Reno" in populations)
print("Carson City" in populations)
print(populations.get("Carson City"))       # None if absent
print(populations.get("Carson City", 0))    # supply a default

True
False
None
0


## III. Useful dictionary methods

`.keys()`, `.values()`, and `.items()` return **view objects**, not lists. They can be looped over, and you can use `list(...)` if you specifically need a list. `.pop()` removes an entry and returns its value; `.update()` adds or replaces entries; `.copy()` makes a  copy.

In [12]:
print(Ctemp.keys())
print(Ctemp.values())
print(Ctemp.items())
print(list(Ctemp.keys()))

removed_temp = Ctemp.pop("Tucson")
print("Removed:", removed_temp)
Ctemp["Tucson"] = 95       # add it back
Ctemp.update({"Bridgeport": 62})
Ntemp = Ctemp.copy()
print("Original:", Ctemp)
print("Copy:", Ntemp)

dict_keys(['Tucson', 'Truckee', 'Reno', 'Laramie', 'Flagstaff', 'Bozeman'])
dict_values([95, 65, 74, 50, 75, 70])
dict_items([('Tucson', 95), ('Truckee', 65), ('Reno', 74), ('Laramie', 50), ('Flagstaff', 75), ('Bozeman', 70)])
['Tucson', 'Truckee', 'Reno', 'Laramie', 'Flagstaff', 'Bozeman']
Removed: 95
Original: {'Truckee': 65, 'Reno': 74, 'Laramie': 50, 'Flagstaff': 75, 'Bozeman': 70, 'Tucson': 95, 'Bridgeport': 62}
Copy: {'Truckee': 65, 'Reno': 74, 'Laramie': 50, 'Flagstaff': 75, 'Bozeman': 70, 'Tucson': 95, 'Bridgeport': 62}


`.copy()` makes a **shallow** copy. Changing a top-level value in the copy will not change the original, but nested mutable objects (such as lists) are still shared. We will see an example of nested dictionaries below.

## IV. Looping through dictionaries

A `for` loop over a dictionary iterates through its **keys**. Use `.values()` for values and `.items()` for both keys and values. To impose a predictable sorted order, use `sorted(d)` or `sorted(d.keys())`.

In [14]:
for city in Ctemp:
    print(city, Ctemp[city])

for temp in Ctemp.values():
    print(f"value is: {temp}")

for city, temp in Ctemp.items():
    print("City:", city, "Temperature:", temp)

for city in sorted(Ctemp):
    print(city, Ctemp[city])

Truckee 65
Reno 74
Laramie 50
Flagstaff 75
Bozeman 70
Tucson 95
Bridgeport 62
value is: 65
value is: 74
value is: 50
value is: 75
value is: 70
value is: 95
value is: 62
City: Truckee Temperature: 65
City: Reno Temperature: 74
City: Laramie Temperature: 50
City: Flagstaff Temperature: 75
City: Bozeman Temperature: 70
City: Tucson Temperature: 95
City: Bridgeport Temperature: 62
Bozeman 70
Bridgeport 62
Flagstaff 75
Laramie 50
Reno 74
Truckee 65
Tucson 95


## V. Building dictionaries from real data

Most of the dictionaries we use will be built while reading files, rather than hardcoded. We first create an empty dictionary, read data line by line, extract the fields we need, and assign values to keys.

### Example 1: FASTA sequence ID as key,  DNA sequence as value

The example below assumes each FASTA record has exactly two lines: one header and one complete sequence line. This is suitable for the short example file but **not** for general FASTA files if sequences occur across multiple lines (with line endings).

In [17]:
ID_Seq = {}
with open("SG_ref.fasta.short") as IN:
    for Line in IN:
        ID = Line.strip("\n") ## reads first line
        Seq = IN.readline().strip("\n") ## reads next line, so each time through for loop, 2 lines get read.
        ID_Seq[ID] = Seq

print("Sequences stored:", len(ID_Seq)) ## len() works on dict just as on list
for seq_id in list(ID_Seq)[:5]:
    print(seq_id, ID_Seq[seq_id][:40])

Sequences stored: 500
>scaffold_1 ATCGGCAGCTACATACCCTCCCTTATCCTGCAAAGCAGCT
>scaffold_2 AGAGAACTGTTCTCAAACCTCGAAGCCATGACTGTCTGCA
>scaffold_3 AGCGTACCGCTGTCATAGTGACCCGGAGCTTTTGATCCGC
>scaffold_4 ATTCGGTCTAAGGAGGCACTCTCGGACTTGCTCTGCCCTT
>scaffold_5 TCGATTCCGCAGTGAGGCTGGAGAGCTTTGTGTTCACCAG


For general FASTA files, the following slightly longer approach handles sequences spanning multiple lines. The sequence for each ID is stored when the next header is encountered. This is a safer approach, because it ensure that only lines starting with ">" are assigned to keys, and only lines starting with [ATCG] are stored as values.

In [20]:
import re

ID_Seq = {}
with open("SG_ref.fasta.short") as IN:
    for line in IN:
        line = line.strip()
        if re.match("^>", line):
            seq_id = line
        elif re.match("^[ATCG]", line):
            ID_Seq[seq_id] = line
print("Sequences stored:", len(ID_Seq))

for ID, Seq in ID_Seq.items():
    print("ID:", ID, "DNA:", Seq)

Sequences stored: 500
ID: >scaffold_1 DNA: ATCGGCAGCTACATACCCTCCCTTATCCTGCAAAGCAGCTGGGATCAATCAAAGTATAAAATCAAAACCTTTTTACAGATCGGAAGA
ID: >scaffold_2 DNA: AGAGAACTGTTCTCAAACCTCGAAGCCATGACTGTCTGCAATGGCGAGGAATCAAACGTGCAGCACAGGAAACACTCTCTGGCTCCT
ID: >scaffold_3 DNA: AGCGTACCGCTGTCATAGTGACCCGGAGCTTTTGATCCGCTGAGCAGAAAGGCTCCTTGTCCCCCTTTCCGAACGGGCTAATGGAGA
ID: >scaffold_4 DNA: ATTCGGTCTAAGGAGGCACTCTCGGACTTGCTCTGCCCTTACAGTGGGGAGGGATTTACCTCCTTGGTCCCCACATAGAGGTTTGGG
ID: >scaffold_5 DNA: TCGATTCCGCAGTGAGGCTGGAGAGCTTTGTGTTCACCAGATGCGCAAGAAAACCAATAGCACGTCCTGTTGTTTGTAAGAACGAGC
ID: >scaffold_6 DNA: GGACGCTCTCGGCTTCAGAACGGCACAACTTCGCTCTCACGGCTCGGAAGTGGCACTCGCGGTGAACTGCTCCGGAGCGCTTGCCGT
ID: >scaffold_7 DNA: ATTTGGTCGGTATGACGTCATATGACGTCTGGATGAGGTCATATGGGGTTTTTTACGTCATTTGGGCAATTCTCCGTCGTTTGGTGCG
ID: >scaffold_8 DNA: TATGCTTTGTGGAAAGCAGCATCTTCTCTTGAAATCTTATCTGTCTTAGGAAAATACAAACACCTTTGTTTCCCAGCTTTGGTGCTGT
ID: >scaffold_9 DNA: CCGGCTGTAGAATCAGTCCTGTAACCATTTTGTTGATTCTCCTGATTCAACTGGCTCGTTCCAACTATTTCCCTTCGGCTGGG

### Example 2: Population metadata

Here we build **two dictionaries** from four columns in `muricata_pops_lat_long.csv`: population → region, and population → latitude. This assumes the file contains four comma-separated fields per row and no header. We convert latitude to a float because it is a number.

In [28]:
pop_dict = {}
lat_dict = {}
with open("muricata_pops_lat_long.csv") as IN:
    for line in IN:
        line = line.strip()
        region, pop, lat, longitude = line.split(",")
        pop_dict[pop] = region
        lat_dict[pop] = float(lat)

print("Populations:", len(pop_dict))
for pop in list(pop_dict)[:5]:
    print(pop, pop_dict[pop], lat_dict[pop])

Populations: 14
CH SCI 34.003334
LO SCI 34.003334
CP SCI 34.013676
RR SOC 34.013676
PB SLO 34.024463


## VI. Counting observations with dictionaries

Counting repeated things is one of the most useful dictionary applications. The **keys** are the things being counted, and the **values** are their counts.

### Example 1: Count bases in a DNA sequence

First, the longer form using an `if` statement:

In [29]:
seq = "ATGCTTACGAATTCGACCCGTAATATATATATAT"
counts = {}
for base in seq:
    if base not in counts: # notice what this does. If base doesnt exist yet as a key, it gets assigned here.
        counts[base] = 1
    else:
        counts[base] += 1
print(counts)

{'A': 12, 'T': 12, 'G': 4, 'C': 6}


We can accomplish the same thing more compactly using `.get(base, 0)`. If the key is absent, its previous count is treated as zero.

In [30]:
counts = {}
for base in seq:
    counts[base] = counts.get(base, 0) + 1
print(counts)
print("Total bases:", sum(counts.values()))

{'A': 12, 'T': 12, 'G': 4, 'C': 6}
Total bases: 34


### Example 2: Count sequencing reads per bird (FASTQ)

The original class example uses `sample_passerina.fastq`, which contains reads from many individual birds. The header lines identify the birds, and we want the number of reads for each bird. FASTQ records have **four lines**; below we process records in groups of four so that a quality-score line starting with `@` is not mistaken for a new read.

**Important:** This code assumes the first whitespace-separated token of each FASTQ header is the bird identifier, as in the course example. Check the header format in any other dataset before using this approach.

In [31]:
id_counts = {}
with open("sample_passerina.fastq") as IN:
    while True:
        header = IN.readline()
        if header == "":
            break
        sequence = IN.readline()
        plus = IN.readline()
        quality = IN.readline()
        if not quality:
            raise ValueError("Incomplete FASTQ record")
        bird_id = header.split()[0].removeprefix("@")
        id_counts[bird_id] = id_counts.get(bird_id, 0) + 1

print("Bird IDs:", len(id_counts))
for bird, count in list(id_counts.items())[:10]:
    print(f"{bird}: {count}")

Bird IDs: 190
NDW_AM_38722: 83
NVP_CY_48141: 33
NVP_CY_48147: 52
SDH_CY_43460: 62
SDC_CY_55118: 45
NEP_AM_55142: 42
NDW_AM_38719: 46
LAA_CY_20575: 110
SDN_AM_43441: 75
WYS_CY_46898: 89


### Example 3: Grouping observations into lists

Sometimes we want to **collect** all observations belonging to a key, not just count them. Here each population maps to a list of sample IDs. `.setdefault(key, [])` creates an empty list only when that key is absent.

In [32]:
samples = [("Reno", "sample_1"), ("Truckee", "sample_2"),
           ("Reno", "sample_3"), ("Reno", "sample_4")]

samples_by_pop = {}
for pop, sample_id in samples:
    samples_by_pop.setdefault(pop, []).append(sample_id)
print(samples_by_pop)

{'Reno': ['sample_1', 'sample_3', 'sample_4'], 'Truckee': ['sample_2']}


## VII. Nested dictionaries

A nested dictionary is simply a dictionary containing other dictionaries as values. It is useful for sample: metadata, gene: annotations, or chromosome: features.

In [33]:
Bengals = {
    'Offense': {
        9:  'Burrow',
        1:  'Chase',
        84: 'Endries',
        75: 'Brown',
        5:  'Higgins'
    },
    'Defense': {
        6: 'Cook',
        23: 'Hill',
        97: 'Dexy',
        0: 'Turner',
        53: 'Mafe'
    }
}

In [35]:
# Accessing element using key 
print("Offense dict: ", Bengals['Offense']) 
print("Offense, number 1: ", Bengals['Offense'][1]) 
print("Defense, number 97: ", Bengals['Defense'][97]) 

#iterating through nested dictionaries
for i in Bengals:
    print(i, Bengals[i])
    print(i, Bengals[i].keys())
    print(i, Bengals[i].values())

Offense dict:  {9: 'Burrow', 1: 'Chase', 84: 'Endries', 75: 'Brown', 5: 'Higgins'}
Offense, number 1:  Chase
Defense, number 97:  Dexy
Offense {9: 'Burrow', 1: 'Chase', 84: 'Endries', 75: 'Brown', 5: 'Higgins'}
Offense dict_keys([9, 1, 84, 75, 5])
Offense dict_values(['Burrow', 'Chase', 'Endries', 'Brown', 'Higgins'])
Defense {6: 'Cook', 23: 'Hill', 97: 'Dexy', 0: 'Turner', 53: 'Mafe'}
Defense dict_keys([6, 23, 97, 0, 53])
Defense dict_values(['Cook', 'Hill', 'Dexy', 'Turner', 'Mafe'])


A biological example: each sample ID is a key, and its value is another dictionary containing population, coverage, and ploidy.

In [36]:
sample_meta = {
    "sample_1": {"population": "Reno", "coverage": 18.2, "ploidy": 2},
    "sample_2": {"population": "Truckee", "coverage": 12.7, "ploidy": 4},
}
print(sample_meta["sample_1"]["coverage"])
for sample_id, metadata in sample_meta.items():
    print(sample_id, metadata["population"], metadata["ploidy"])

18.2
sample_1 Reno 2
sample_2 Truckee 4


## VIII. Dictionaries in bioinformatic workflows

Biological data is naturally keyed: gene → coordinates, transcript → exons, sequence ID → sequence, feature type → count, and sample → summary statistics. The following examples use `Saccharomyces_cerevisiae.R64-1-1.92.chromosome.II.gff3` from the course files.

GFF3 columns are tab-separated. Column 3 identifies the feature type, columns 4 and 5 give coordinates, and column 9 contains semicolon-separated attributes. We skip comment lines and malformed records.

### Example 1: Gene ID → coordinates

In [39]:
gff_file = "Saccharomyces_cerevisiae.R64-1-1.92.chromosome.II.gff3"
gene_coords = {}
with open(gff_file) as IN:
    for line in IN:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        if len(fields) < 9 or fields[2] != "gene":
            continue
        chrom, start, end = fields[0], int(fields[3]), int(fields[4])
        gene_id = None
        for item in fields[8].split(";"):
            if item.startswith("ID="):
                gene_id = item.split("=", 1)[1]
                break
        if gene_id is not None:
            gene_coords[gene_id] = (chrom, start, end)

print("Genes:", len(gene_coords))
for gene_id, coords in list(gene_coords.items())[:5]:
    print(gene_id, coords)

Genes: 462
gene:YBL113C ('II', 280, 2658)
gene:YBL113W-A ('II', 646, 1128)
gene:YBL112C ('II', 2582, 2899)
gene:YBL111C ('II', 2907, 5009)
gene:YBL109W ('II', 5790, 6125)


### Example 2: Filter genes by length

Because GFF3 coordinates are inclusive, length is `end - start + 1`. We can create a second dictionary containing only genes longer than 2,000 bp.

In [40]:
long_genes = {}
for gene_id, (chrom, start, end) in gene_coords.items():
    if end - start + 1 > 2000:
        long_genes[gene_id] = (chrom, start, end)
print("Genes longer than 2000 bp:", len(long_genes))

Genes longer than 2000 bp: 100


### Example 3: Count features by type

This applies the same counting pattern we used for bases and bird IDs, but now the keys are GFF3 feature types.

In [41]:
feature_counts = {}
with open(gff_file) as IN:
    for line in IN:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        if len(fields) < 9:
            continue
        feature = fields[2]
        feature_counts[feature] = feature_counts.get(feature, 0) + 1

for feature in sorted(feature_counts):
    print(feature, feature_counts[feature])

CDS 492
chromosome 1
exon 510
gene 462
mRNA 462
ncRNA 1
ncRNA_gene 17
snRNA 1
snoRNA 2
tRNA 13


### Example 4: Feature type → list of lengths

Instead of counting each feature, we can collect the lengths of all features of each type. This is useful for plotting distributions later.

In [42]:
lengths = {}
with open(gff_file) as IN:
    for line in IN:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        if len(fields) < 9:
            continue
        feature = fields[2]
        length = int(fields[4]) - int(fields[3]) + 1
        lengths.setdefault(feature, []).append(length)

print("Gene lengths collected:", len(lengths.get("gene", [])))

Gene lengths collected: 462


### Example 5: Transcript or gene parent → CDS intervals

The `Parent=` attribute connects a CDS feature to its parent annotation. We store a list of `(start, end)` tuples for each parent ID. Parent identifiers in GFF3 can refer to transcripts rather than genes, depending on the annotation.

In [43]:
cds_dict = {}
with open(gff_file) as IN:
    for line in IN:
        if line.startswith("#"):
            continue
        fields = line.strip().split("\t")
        if len(fields) < 9 or fields[2] != "CDS":
            continue
        start, end = int(fields[3]), int(fields[4])
        for item in fields[8].split(";"):
            if item.startswith("Parent="):
                for parent in item.split("=", 1)[1].split(","):
                    cds_dict.setdefault(parent, []).append((start, end))

for parent, intervals in list(cds_dict.items())[:5]:
    print(parent, intervals)

transcript:YBL113C [(280, 2658)]
transcript:YBL113W-A [(646, 1128)]
transcript:YBL112C [(2582, 2899)]
transcript:YBL111C [(2907, 4116), (4216, 5009)]
transcript:YBL109W [(5790, 6125)]


## IX. Summary

The most useful patterns to remember are:

- `d[key] = value` to build a dictionary while reading data.
- `d.get(key, 0) + 1` to count repeated observations.
- `for key, value in d.items():` to work with both parts of a pair.
- `sorted(d)` to visit keys in sorted order.
- `d.setdefault(key, []).append(value)` to group multiple observations under one key.

---
# Appendix: Dictionary reference tables

These tables are intended as a reference when working on assignments and scripts. The examples use small dictionaries such as `counts = {"A": 5, "C": 3}` and `meta = {"s1": "Reno"}`.

## A. Creating, accessing, and modifying dictionaries

| Method / pattern | What it does | Example | Biological use / note |
|---|---|---|---|
| `{}` | Create empty dictionary | `counts = {}` | Start a base-count dictionary |
| `dict()` | Another way to create empty dictionary | `counts = dict()` | Equivalent to `{}` |
| `{"A": 5, "C": 3}` | Create with entries | `counts = {"A": 5, "C": 3}` | Base counts |
| `dict(zip(keys, values))` | Pair two iterables | `dict(zip(["A", "C"], [5, 3]))` | Map IDs to measurements; `zip` stops at shortest input |
| `d[key]` | Retrieve value; raises `KeyError` if missing | `counts["A"]` | Retrieve base count |
| `d.get(key)` | Retrieve or return `None` if missing | `counts.get("G")` | Optional annotation |
| `d.get(key, default)` | Retrieve or return default | `counts.get("G", 0)` | Missing count as zero |
| `d[key] = value` | Add or overwrite entry | `counts["G"] = 2` | Store gene coverage |
| `d.update(other)` | Add/overwrite several entries | `counts.update({"T": 4})` | Add more metadata; overlapping keys overwritten, **not summed** |
| `d.setdefault(key, default)` | Return existing value, or insert default | `counts.setdefault("N", 0)` | Initialize missing category |
| `d.pop(key)` | Remove key and return its value; errors if absent | `counts.pop("A")` | Remove unwanted entry |
| `d.pop(key, default)` | Remove key, or return default if absent | `counts.pop("N", 0)` | Remove safely |
| `del d[key]` | Delete an entry; errors if absent | `del counts["C"]` | Remove failed sample |
| `d.popitem()` | Remove and return last inserted pair | `counts.popitem()` | Useful for some workflows; not sorted order |
| `d.clear()` | Remove all entries | `counts.clear()` | Reset dictionary |
| `d.copy()` | Make shallow copy | `new_counts = counts.copy()` | Preserve original top-level entries |
| `len(d)` | Number of pairs | `len(counts)` | Number of distinct bases observed |
| `key in d` | Check whether key exists | `"A" in counts` | Test whether gene was observed |
| `key not in d` | Test absence | `"N" not in counts` | Identify unseen category |

## B. Views, loops, and sorting

| Method / pattern | What it does | Example | Biological use / note |
|---|---|---|---|
| `d.keys()` | View of keys | `counts.keys()` | Sequence IDs; not a list |
| `d.values()` | View of values | `counts.values()` | Read counts; not a list |
| `d.items()` | View of `(key, value)` pairs | `counts.items()` | Gene → measurement pairs |
| `list(d.keys())` | Convert keys to list | `list(counts.keys())` | Save IDs as a list |
| `for k in d:` | Loop through keys | `for k in counts: print(k)` | Visit gene IDs |
| `for k in d.keys():` | Explicit loop through keys | `for k in counts.keys(): print(k)` | Same as `for k in d` |
| `for v in d.values():` | Loop through values | `for v in counts.values(): print(v)` | Examine read depths |
| `for k, v in d.items():` | Loop through key/value pairs | `for k, v in counts.items(): print(k, v)` | Print sample and count |
| `sorted(d)` | Sorted list of keys | `sorted(counts)` | Alphabetical base names |
| `for k in sorted(d):` | Visit keys in sorted order | `for k in sorted(counts): print(k, counts[k])` | Reproducible output |
| `sorted(d.items())` | Sort pairs by key | `sorted(counts.items())` | Ordered summary |
| `sorted(d.items(), key=lambda x: x[1])` | Sort pairs by value | `sorted(counts.items(), key=lambda x: x[1])` | Rank counts low to high |
| `sum(d.values())` | Sum numeric values | `sum(counts.values())` | Total bases or reads |
| `max(d, key=d.get)` | Key with largest numeric value | `max(counts, key=counts.get)` | Most frequent base; requires nonempty dictionary |

## C. Common patterns for biological data

| Task | Example | Explanation |
|---|---|---|
| Count bases | `counts[b] = counts.get(b, 0) + 1` | Increment count for base `b` |
| Count codons | `codons[codon] = codons.get(codon, 0) + 1` | Count each codon |
| Count features | `features[ftype] = features.get(ftype, 0) + 1` | Count GFF3 gene, CDS, etc. |
| FASTA ID → sequence | `fasta[seq_id] = seq` | Retrieve a sequence by ID |
| Sample → population | `sample_pop[sample] = pop` | Join metadata by sample ID |
| Population → latitude | `latdict[pop] = float(lat)` | Store numerical site metadata |
| Sample → metadata | `meta[sample] = {"pop": pop, "depth": depth}` | Nested dictionary |
| Retrieve nested value | `meta[sample]["depth"]` | Get coverage for one sample |
| Group samples by population | `groups.setdefault(pop, []).append(sample)` | Each value is a list |
| Group CDS intervals | `cds.setdefault(parent, []).append((start, end))` | One parent can have many intervals |
| Filter dictionary | `{k: v for k, v in counts.items() if v > 2}` | Keep entries above threshold (dictionary comprehension) |
| Transform values | `{k: v / 100 for k, v in counts.items()}` | Rescale numeric values |
| Build dictionary from pairs | `dict([("A", 5), ("C", 3)])` | Convert list of pairs |
| Merge two dictionaries | `merged = {**d1, **d2}` | Keys in `d2` replace matching keys in `d1` |

## D. Common mistakes and important details

| Issue | Example | What to remember |
|---|---|---|
| Missing key | `counts["N"]` | Raises `KeyError` if `"N"` absent; consider `.get()` |
| Duplicate key | `{"A": 1, "A": 2}` | Only one `"A"` key remains, with value `2` |
| Keys must be hashable | `d[[1, 2]] = 3` | A list cannot be a dictionary key; a suitable tuple can |
| Insertion order vs sorted order | `list(d)` vs `sorted(d)` | Insertion order is preserved; use `sorted` for alphabetical order |
| Views are not lists | `d.keys()` | Use `list(d.keys())` if you need list operations |
| Aliasing | `d2 = d1` | Both names refer to same dictionary; use `.copy()` for shallow copy |
| Nested shallow copy | `d2 = d1.copy()` | Nested lists/dictionaries are still shared |
| Overwriting while counting | `counts[b] = 1` | Resets count every time; use `counts.get(b, 0) + 1` |
| Iterating while changing size | `for k in d: del d[k]` | Avoid adding/removing keys while iterating; loop over `list(d)` instead |
| `.update()` and counts | `d1.update(d2)` | Overwrites shared keys; does **not** add their counts |
| Strings vs numbers | `latdict[pop] = lat` | File fields are strings until converted with `float()` or `int()` |
| FASTA line wrapping | `Seq = IN.readline()` | Only reads one line; use multi-line parsing for general FASTA |
| FASTQ record structure | Four lines per record | Do not treat every line beginning with `@` as a header |

**Remember:** Dictionary methods work on dictionary objects. `sorted(d)` and `len(d)` are built-in Python functions, not dictionary methods.